# distillPDF — OCR for scanned PDFs

Some PDFs are just scanned images — there's no text to extract. distillPDF finds those pages, OCRs them, and folds the recovered text (and, on the accurate tier, **tables**) into distillPDF's **HTML**, **Markdown**, and a **searchable PDF**. Born-digital pages keep their normal extraction.

**Two tiers:**
- **fast** (default) — a **bundled Tesseract** engine: works on a plain `pip install distillpdf`, no extra, no download, fully offline. English/Portuguese/Norwegian in the wheel, with the document language auto-detected up front. Flat text (no tables), ~0.8 s/page.
- **accurate** — the **granite-docling** VLM: structure + tables, ~6 s/page. Needs a model runtime you install yourself (MLX on Apple Silicon, PyTorch on Windows/Linux); see `distillpdf.ocr.install_help('granite')` or docs/ocr-setup.md.

The workflow lives on the document: **open → `run_ocr` → write**; OCR runs once and is cached, with a progress bar on a terminal.

## Install

In [ ]:
# Fast tier needs nothing beyond distillpdf. For more languages or the accurate tier:
# accurate tier: install a runtime — print(distillpdf.ocr.install_help('granite'))
import distillpdf
print('distillpdf', distillpdf.__version__)
print('OCR engines:', [d.name for d in distillpdf.ocr.backend_descriptors()])

## Quickstart

`run_ocr()` OCRs every scanned page once (downloading the model on first use) and caches the
result on the document. The `to_*` methods then fold that text into their output — no second
model pass, no DocTags to handle.

In [ ]:
doc = distillpdf.open("scanned.pdf")     # <-- your scanned / image-only PDF

# OCR every scanned page once (cached on the document). First run downloads the model
# (a few hundred MB); a progress bar is shown automatically. Trying it out? Start small
# with e.g. only={1, 2, 3} before committing to a long document.
doc.run_ocr()

doc.to_pdf("scanned.searchable.pdf")     # searchable PDF
doc.to_html("scanned.html")              # clean HTML
doc.to_markdown("scanned.md")            # Markdown

For a **single** output you can skip the explicit `run_ocr()` and pass `ocr=True` — it runs OCR once, then writes:

```python
doc.to_pdf("scanned.searchable.pdf", ocr=True)
```

## Command line — no Python needed

`--ocr` does open → OCR (progress bar shown automatically) → write:

```bash
distillpdf scan.pdf --ocr                  # -> scan.searchable.pdf  (keeps scan + hidden text layer)
distillpdf scan.pdf --ocr --remove-raster  # -> reflowed clean text + figures, much smaller file
distillpdf scan.pdf --ocr -o out.html      # OCR'd HTML  (use a .md path for Markdown)
distillpdf *.pdf      --ocr -o out/        # batch: out/<name>.searchable.pdf per input
```

If you render **before** OCR, the scanned pages come out (almost) empty — so the render methods
emit a warning telling you to call `run_ocr()`:

In [ ]:
import warnings
doc2 = distillpdf.open("scanned.pdf")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    doc2.to_html(return_string=True)
    print(w[0].message if w else 'no scanned pages')

## Two kinds of searchable PDF

By default the original scan is **kept** and the OCR text is added as an *invisible, selectable*
layer over it — you always see the original page, so OCR mistakes never hide content (best for
archives / legal). Pass `remove_raster=True` to instead reflow to clean text + cropped figures
and drop the scan (a much smaller file).

In [ ]:
doc.to_pdf("keep.pdf")                        # keep scan + hidden text layer (default)
doc.to_pdf("clean.pdf", remove_raster=True)   # reflow to text, drop the raster

## Which pages will be OCR'd?

`ocr_plan()` reports, per page, whether it needs OCR (image-only / sparse text) or already has
a good text layer (born-digital, left untouched).

In [ ]:
for p in doc.ocr_plan()[:8]:
    print(f"page {p['page']:>3}  needs_ocr={p['needs_ocr']!s:<5}  {p['reason']}")

## Advanced — bring your own OCR model

A backend is just *image bytes → a [DocTags](https://github.com/docling-project/docling) string*;
distillPDF renders the DocTags into HTML / PDF / Markdown. Subclass `OcrBackend` to plug in any
model or service, then hand it to `run_ocr`. The default backend is configurable too (model
cache dir, HF token, device).

In [ ]:
from distillpdf.ocr import OcrBackend

class MyBackend(OcrBackend):
    name = "my-backend"
    def ocr_page(self, image: bytes) -> str:
        # send `image` (PNG/JPEG bytes) to your model, return its DocTags string
        ...

# doc.run_ocr(MyBackend())

# Configure the default backend:
backend = distillpdf.get_backend("granite-docling", model_dir=None, hf_token=None, device="auto")

**Zero-Python-dep path.** distillPDF also ships a pure-Rust `ServerOcrEngine` that talks to a
llama.cpp `llama-server` you run yourself — no `[ocr]` extra needed.

**Under the hood.** If you already have DocTags, the pure-Rust renderers are exposed directly:
`distillpdf.ocr.render_doctags(dt)` (→ HTML) and `distillpdf._distillpdf.html_to_markdown(html)`.

---
See the [README](../README.md) for the full API.